# Convert fine-tuned CV models to quantized GGUF — Colab GPU

Run this **once** on a Colab GPU runtime. It merges your LoRA adapters into
their base models, converts to GGUF, quantizes to **Q4_K_M**, and uploads
the finished files directly to your Hugging Face Hub repo.

**Outputs:**
- `phi3-cv-Q4_K_M.gguf` (~2.3 GB) — fine-tuned Phi-3 CV writer

CV analysis judging runs on Modal GPU — see `cv_analysis/infra/gpu_inference/README.md`.
**Setup:**
1. *Runtime → Change runtime type → T4 GPU*
2. Run cells top to bottom — each cell must finish before the next.

In [ ]:
# Cell 1 — Confirm GPU is attached
!nvidia-smi -L

In [ ]:
# Cell 2 — Install dependencies (no bitsandbytes needed)
!pip install -q transformers peft accelerate huggingface-hub sentencepiece
# Colab ships torchao 0.10.0 but peft requires >=0.16.0.
# --no-deps prevents torchao from pulling in a different torch version.
!pip install -q "torchao>=0.16.0" --no-deps
# Re-pin protobuf LAST — tensorflow (pre-installed on Colab) downgrades it to
# an old version that lacks `runtime_version`, breaking peft's import chain.
!pip install -q "protobuf>=4.25" --force-reinstall
!apt-get -qq install -y cmake build-essential
print('Done.')

In [ ]:
# Cell 3 — Hugging Face token
# Needs READ access to gated repos + WRITE access to upload GGUFs.
# Create a token at https://huggingface.co/settings/tokens (role: write)
import os, getpass
TOKEN = getpass.getpass('Paste your HF token (hidden): ').strip()
os.environ['HF_TOKEN'] = TOKEN
assert TOKEN, 'Token is empty — paste a valid HF token above.'
print('Token saved.')

In [ ]:
# Cell 4 — Configuration
import os

WORK_DIR = '/content'
OUT_DIR  = f'{WORK_DIR}/gguf'
LLAMA_DIR = f'{WORK_DIR}/llama.cpp'
os.makedirs(OUT_DIR, exist_ok=True)

# Models to merge and convert
# (base_model, lora_adapter, adapter_subfolder, output_stem)
FINE_TUNES = {
    'phi3': (
        'microsoft/Phi-3-mini-4k-instruct',
        'basmalaalaa029/phi3-cv',
        'checkpoint-200',
        'phi3-cv',
    ),
}
# Q8_0 for phi3 — Q4_K_M destroys quality on small (3.8B) fine-tuned models.
QUANT_PER_KEY = {
    'phi3': 'Q8_0',    # ~3.7 GB — safe for fine-tuned Phi-3-mini
}
QUANT = 'Q4_K_M'  # fallback for any key not in QUANT_PER_KEY

# Hugging Face repo where the finished GGUFs will be uploaded
# The repo is created automatically if it does not exist yet.
HF_UPLOAD_REPO = 'basmalaalaa029/cv-agent-gguf'  # change if needed

print('Config ready.')
print(f'  Output dir : {OUT_DIR}')
print(f'  Upload repo: {HF_UPLOAD_REPO}')

In [ ]:
# Cell 5 — merge() helper  (plain fp16, works for both Phi-3 and Qwen)
import os, shutil, torch
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import hf_hub_download

# Patch: some transformers builds ship Qwen2's ROPE_INIT_FUNCTIONS without a
# 'default' entry, causing KeyError when rope_scaling.rope_type == 'default'.
# We inject the standard (unscaled) RoPE initializer if it is missing.
try:
    from transformers.models.qwen2 import modeling_qwen2 as _qwen2_mod
    if 'default' not in _qwen2_mod.ROPE_INIT_FUNCTIONS:
        try:
            from transformers.modeling_rope_utils import _compute_default_rope_parameters
        except ImportError:
            def _compute_default_rope_parameters(config, device, **kw):
                base = getattr(config, 'rope_theta', 10000.0)
                head_dim = getattr(config, 'head_dim',
                                   config.hidden_size // config.num_attention_heads)
                dim = int(head_dim * getattr(config, 'partial_rotary_factor', 1.0))
                inv_freq = 1.0 / (base ** (
                    torch.arange(0, dim, 2, dtype=torch.int64).float().to(device) / dim
                ))
                return inv_freq, 1.0
        _qwen2_mod.ROPE_INIT_FUNCTIONS['default'] = _compute_default_rope_parameters
        print("  [patch] ROPE_INIT_FUNCTIONS['default'] injected into qwen2 modeling")
except Exception as _e:
    print(f'  [warn] rope patch skipped: {_e}')

merged = {}   # filled by cells 6 and 7

def merge(base, adapter, subfolder, dest):
    """Load base in fp16, apply LoRA, merge weights, save to dest."""
    if os.path.isdir(dest):
        print(f'[skip] {dest} already exists — delete it to re-merge.')
        return

    print(f'Loading base: {base}')
    config = AutoConfig.from_pretrained(base, token=TOKEN, trust_remote_code=True)

    # Strip fields that block plain fp16 loading.
    # Only clear rope_scaling for Phi-3's broken 'su'/'longrope' types.
    # Qwen2.5 uses 'yarn' which must be kept or the RoPE init fails with KeyError.
    rs = getattr(config, 'rope_scaling', None)
    if rs is not None:
        rope_type = rs.get('rope_type') or rs.get('type', '')
        if rope_type.lower() in ('su', 'longrope', 'default', ''):
            config.rope_scaling = None
            print(f'  [patch] rope_scaling cleared (type={rope_type!r})')
        else:
            # keep but add old 'type' alias so modeling_phi3.py doesn't crash
            config.rope_scaling['type'] = rope_type
            print(f'  [patch] rope_scaling kept + type alias added (type={rope_type!r})')
    if getattr(config, 'quantization_config', None) is not None:
        delattr(config, 'quantization_config')
        print('  [patch] quantization_config removed')
    # Newer transformers' torch_dtype setter requires pad_token_id to be present;
    # ensure it is set so the deprecated setter path doesn't raise AttributeError.
    if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
        config.pad_token_id = getattr(config, 'eos_token_id', 0)

    model = AutoModelForCausalLM.from_pretrained(
        base,
        config=config,
        dtype=torch.float16,        # 'dtype' replaces deprecated 'torch_dtype'
        trust_remote_code=True,
        token=TOKEN,
        attn_implementation='eager',
    )

    kw = {'token': TOKEN}
    if subfolder:
        kw['subfolder'] = subfolder
    print(f'Applying LoRA: {adapter}' + (f' [{subfolder}]' if subfolder else ''))
    model = PeftModel.from_pretrained(model, adapter, **kw)

    print('Merging ...')
    model = model.merge_and_unload()

    # Fix _tied_weights_keys type mismatch (some models save it as list)
    for m in model.modules():
        if isinstance(getattr(m, '_tied_weights_keys', None), list):
            m._tied_weights_keys = {}

    print(f'Saving to {dest} ...')
    model.save_pretrained(dest, safe_serialization=True)
    AutoTokenizer.from_pretrained(
        base, trust_remote_code=True, token=TOKEN
    ).save_pretrained(dest)

    # Phi-3 needs tokenizer.model (SentencePiece) which AutoTokenizer may not write
    sp_dst = os.path.join(dest, 'tokenizer.model')
    if not os.path.exists(sp_dst):
        try:
            sp_src = hf_hub_download(repo_id=base, filename='tokenizer.model', token=TOKEN)
            shutil.copy(sp_src, sp_dst)
            print('  [fix] tokenizer.model copied from HF hub')
        except Exception as e:
            print(f'  [warn] tokenizer.model not fetched: {e}')

    # ── Sanity-check the merged fp16 model before paying the GGUF conversion cost ──
    # If this outputs gibberish, the LoRA merge itself is broken (not quantization).
    print('  [verify] Quick generation test on merged fp16 model...')
    try:
        _tok = AutoTokenizer.from_pretrained(dest, trust_remote_code=True)
        _inp = _tok('Name: Sara\nRole: Software Engineer\nWrite a one-line professional summary:',
                    return_tensors='pt').to(model.device)
        with torch.no_grad():
            _out = model.generate(**_inp, max_new_tokens=40, do_sample=False,
                                  pad_token_id=_tok.eos_token_id, use_cache=False)
        _text = _tok.decode(_out[0][_inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        print(f'  [verify] Output: {repr(_text[:200])}')
        # Heuristic: a coherent output has mostly ASCII letters and spaces
        _ascii_ratio = sum(c.isascii() and (c.isalpha() or c == ' ') for c in _text) / max(len(_text), 1)
        if _ascii_ratio < 0.5 or len(set(_text.split())) < 3:
            print('  [WARN] Output looks like garbage! Stop here and check the LoRA merge.')
            print('         Converting to GGUF will NOT fix a broken merge.')
            raise RuntimeError('Merged model failed sanity check — GGUF conversion skipped.')
        else:
            print('  [verify] Output looks coherent — proceeding to GGUF conversion.')
    except RuntimeError:
        raise
    except Exception as _ve:
        print(f'  [warn] Verification skipped: {_ve}')

    del model
    torch.cuda.empty_cache()
    print(f'Done -> {dest}\n')

print('merge() helper ready.')

In [ ]:
# Cell 6b — DIAGNOSTIC: force re-merge phi3 and inspect LoRA config
# Run this INSTEAD of cell 6 to see exactly what the LoRA adapter contains
# and whether the fp16 merged model is coherent BEFORE wasting time on GGUF.

import shutil, os, torch, json
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
from peft import PeftConfig, PeftModel

key = 'phi3'
base, adapter, sub, stem = FINE_TUNES[key]
dest = f'{WORK_DIR}/merged-{stem}'

# Force delete old merge so verification actually runs
if os.path.isdir(dest):
    print(f'Deleting old merge at {dest} ...')
    shutil.rmtree(dest)
    print('Deleted. Will re-merge now.\n')

# Step 1: Inspect the LoRA config BEFORE loading
print('=== LoRA adapter config ===')
try:
    peft_cfg = PeftConfig.from_pretrained(adapter, subfolder=sub or None, token=TOKEN)
    cfg_dict = peft_cfg.to_dict() if hasattr(peft_cfg, 'to_dict') else vars(peft_cfg)
    for k, v in cfg_dict.items():
        print(f'  {k}: {v}')
except Exception as e:
    print(f'  [warn] could not load peft config: {e}')
print()

# Step 2: Load base + adapter and test fp16 BEFORE any GGUF conversion
print('=== Loading base model ===')
config = AutoConfig.from_pretrained(base, token=TOKEN, trust_remote_code=True)
rs = getattr(config, 'rope_scaling', None)
if rs is not None:
    rope_type = rs.get('rope_type') or rs.get('type', '')
    if rope_type.lower() in ('su', 'longrope', 'default', ''):
        config.rope_scaling = None
        print(f'  [patch] rope_scaling cleared (type={rope_type!r})')
    else:
        config.rope_scaling['type'] = rope_type
        print(f'  [patch] rope_scaling kept + type alias added (type={rope_type!r})')
if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
    config.pad_token_id = getattr(config, 'eos_token_id', 0)

model = AutoModelForCausalLM.from_pretrained(
    base, config=config, dtype=torch.float16,
    trust_remote_code=True, token=TOKEN, attn_implementation='eager',
)
print('Base loaded.')

print(f'=== Applying LoRA [{sub}] ===')
kw = {'token': TOKEN}
if sub:
    kw['subfolder'] = sub
model = PeftModel.from_pretrained(model, adapter, **kw)
print('LoRA applied. Merging ...')
model = model.merge_and_unload()
print('Merged.')

# Step 3: Test the MERGED fp16 model directly — this is the ground truth
print('\n=== fp16 inference test (BEFORE any GGUF conversion) ===')
tok = AutoTokenizer.from_pretrained(base, token=TOKEN, trust_remote_code=True)
test_prompt = 'You are an expert CV writer.\n\nWrite a one-line professional summary for a software engineer named Sara:'
inputs = tok(test_prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=60, do_sample=False,
                         pad_token_id=tok.eos_token_id, repetition_penalty=1.1,
                         use_cache=False)
generated = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print(f'Output: {repr(generated[:300])}')

ascii_ratio = sum(c.isascii() and (c.isalpha() or c in ' .,!') for c in generated) / max(len(generated), 1)
print(f'ASCII ratio: {ascii_ratio:.2f}')
if ascii_ratio < 0.5 or len(set(generated.split())) < 3:
    print('\n[RESULT] fp16 model is BROKEN — the LoRA merge is the problem.')
    print('Do NOT proceed to GGUF conversion. The adapter may use a non-standard LoRA variant.')
    print('Check the lora_ga_config and use_bdlora fields printed above.')
else:
    print('\n[RESULT] fp16 model looks GOOD. Proceed to save and then GGUF conversion.')
    # Save for GGUF conversion
    for m in model.modules():
        if isinstance(getattr(m, '_tied_weights_keys', None), list):
            m._tied_weights_keys = {}
    model.save_pretrained(dest, safe_serialization=True)
    tok.save_pretrained(dest)
    import shutil as _sh
    sp_dst = os.path.join(dest, 'tokenizer.model')
    if not os.path.exists(sp_dst):
        from huggingface_hub import hf_hub_download
        try:
            sp_src = hf_hub_download(repo_id=base, filename='tokenizer.model', token=TOKEN)
            _sh.copy(sp_src, sp_dst)
            print('  tokenizer.model copied.')
        except Exception as e:
            print(f'  [warn] tokenizer.model not fetched: {e}')
    merged[key] = (dest, stem)
    print(f'Saved to {dest}')

del model; torch.cuda.empty_cache()

In [ ]:
# Cell 6 — Merge Phi-3 CV writer
# microsoft/Phi-3-mini-4k-instruct  +  basmalaalaa029/phi3-cv  (checkpoint-200)
key = 'phi3'
base, adapter, sub, stem = FINE_TUNES[key]
dest = f'{WORK_DIR}/merged-{stem}'
merge(base, adapter, sub, dest)
merged[key] = (dest, stem)

In [ ]:
# Cell 7 — (removed) CV analysis judge runs on Modal GPU — see cv_analysis/infra/gpu_inference/README.md
print('Skip — ATS judge is deployed via Modal, not GGUF.')

In [ ]:
# Cell 8 — Build llama.cpp  (convert_hf_to_gguf.py + llama-quantize binary)
import subprocess, sys, os

if not os.path.exists(f'{LLAMA_DIR}/.git'):
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/ggerganov/llama.cpp', LLAMA_DIR],
        check=True,
    )
else:
    print('llama.cpp already cloned — skipping clone.')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     '-r', f'{LLAMA_DIR}/requirements.txt'],
    check=True,
)
subprocess.run(
    ['cmake', '-B', f'{LLAMA_DIR}/build', '-S', LLAMA_DIR, '-DLLAMA_CURL=OFF'],
    check=True,
)
# -j2 avoids OOM on Colab's 12 GB RAM
subprocess.run(
    ['cmake', '--build', f'{LLAMA_DIR}/build',
     '--config', 'Release', '-j2', '--target', 'llama-quantize'],
    check=True,
)
print('llama.cpp build complete.')

In [ ]:
# Cell 9 — Convert and quantize all merged models to Q4_K_M GGUF
import subprocess, sys, os, glob

def _qbin():
    for p in [
        f'{LLAMA_DIR}/build/bin/llama-quantize',
        f'{LLAMA_DIR}/build/llama-quantize',
    ]:
        if os.path.exists(p):
            return p
    raise RuntimeError('llama-quantize not found — did cell 8 finish?')

QBIN   = _qbin()
CVTPY  = f'{LLAMA_DIR}/convert_hf_to_gguf.py'

def _run(cmd):
    r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
    if r.returncode != 0:
        raise RuntimeError(f'Command failed:\n  {" ".join(cmd)}')

def convert_and_quantize(key):
    dest, stem = merged[key]
    quant = QUANT_PER_KEY.get(key, QUANT)
    f16   = f'{OUT_DIR}/{stem}-f16.gguf'
    final = f'{OUT_DIR}/{stem}-{quant}.gguf'
    print(f'=== {key}: converting to fp16 GGUF ===')
    _run([sys.executable, CVTPY, dest, '--outfile', f16, '--outtype', 'f16'])
    print(f'=== {key}: quantizing to {quant} ===')
    _run([QBIN, f16, final, quant])
    os.remove(f16)
    print(f'OK  {final}  ({os.path.getsize(final)/1e9:.1f} GB)\n')

for k in merged:
    convert_and_quantize(k)

print('All GGUFs ready:')
for f in sorted(glob.glob(f'{OUT_DIR}/*.gguf')):
    print(f'  {f}  ({os.path.getsize(f)/1e9:.1f} GB)')

In [ ]:
# Cell 10 — Upload GGUFs to Hugging Face Hub
import glob, os
from huggingface_hub import HfApi

api = HfApi(token=TOKEN)

try:
    api.create_repo(repo_id=HF_UPLOAD_REPO, repo_type='model', exist_ok=True)
    print(f'Repo: https://huggingface.co/{HF_UPLOAD_REPO}')
except Exception as e:
    print(f'[warn] repo create: {e}')

gguf_files = sorted(glob.glob(f'{OUT_DIR}/*.gguf'))
assert gguf_files, f'No GGUF files in {OUT_DIR} — run cell 9 first.'

for path in gguf_files:
    name = os.path.basename(path)
    size = os.path.getsize(path) / 1e9
    print(f'Uploading {name} ({size:.1f} GB) ...')
    api.upload_file(
        path_or_fileobj=path,
        path_in_repo=name,
        repo_id=HF_UPLOAD_REPO,
        repo_type='model',
        commit_message=f'Add {name} (Q4_K_M)',
    )
    print(f'  uploaded -> https://huggingface.co/{HF_UPLOAD_REPO}/blob/main/{name}')

print('\nAll uploads complete.')

## Next: pull GGUFs to your local machine and start the server

Run the following once on your **CPU machine** after cell 10 finishes:

```bash
cd ~/Documents/front-end-/ai-models

# Download your fine-tuned GGUFs from HF Hub
python - <<'EOF'
from huggingface_hub import hf_hub_download
import os, shutil

REPO  = 'basmalaalaa029/cv-agent-gguf'   # must match HF_UPLOAD_REPO in cell 4
DEST  = 'models/gguf'
FILES = ['phi3-cv-Q8_0.gguf']

os.makedirs(DEST, exist_ok=True)
for fname in FILES:
    print(f'Downloading {fname} ...')
    src = hf_hub_download(repo_id=REPO, filename=fname)
    shutil.copy(src, os.path.join(DEST, fname))
    print(f'  -> {DEST}/{fname}')
EOF

# CV analysis uses Modal GPU — see cv_analysis/infra/gpu_inference/README.md

# Start the AI server
./scripts/run.sh
```

You should see `[cv] Loading Phi-3 writer (GGUF): phi3-cv-Q4_K_M.gguf` in the
logs with no errors, and CV generation will complete in **1–3 minutes** on CPU.